# 01 — Prepare routing inputs

Build the one-row-per-active-cell routing input from the revised 100 m foundation. A cell is active when it has positive population in any quarter or contains a firm.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

def discover_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ANAL").is_dir() and (candidate / "TOOLS").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or a subdirectory containing ANAL/ and TOOLS/.")

# Local configuration: change only these paths when transferring the project to the air-gapped PC.
PROJECT_DIR = discover_project_dir()
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
RASTER_PATH = ANAL_DATA / "raster_100m_styria.geoparquet"
FIRMS_PATH = ANAL_DATA / "firms_assigned_100m.geoparquet"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
OUTPUT_PATH = ANAL_DATA / "routing" / "inputs" / "active_routing_cells_100m.parquet"
CRS_ANALYSIS = "EPSG:3035"
CRS_ROUTING = "EPSG:4326"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
required_inputs = [RASTER_PATH, FIRMS_PATH, PANEL_PATH]
missing_inputs = [path for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Offline preflight failed; missing local inputs: {missing_inputs}")

In [ ]:
raster = gpd.read_parquet(RASTER_PATH).to_crs(CRS_ANALYSIS)
panel = pd.read_parquet(PANEL_PATH, columns=["grid_id", "population_backcast", "active_firms_t", "active_firms_tminus1"])
firms = gpd.read_parquet(FIRMS_PATH)

if raster["grid_id"].duplicated().any():
    raise ValueError("Raster grid_id must be unique")
if raster["municipality_id"].isna().any():
    raise ValueError("Every raster cell must have a municipality_id")
out_of_scope_firms = firms[firms["municipality_id"].isna()].copy()
in_scope_firms = firms[firms["municipality_id"].notna()].copy()
if in_scope_firms["grid_id_100m"].isna().any():
    raise ValueError("In-scope firm-to-grid assignment is incomplete")
if len(out_of_scope_firms):
    print(f"Excluding {len(out_of_scope_firms):,} firm rows without municipality/grid assignment; these are outside the raster study area.")

panel_active = panel.loc[
    panel[["population_backcast", "active_firms_t", "active_firms_tminus1"]].fillna(0).gt(0).any(axis=1),
    "grid_id",
]
firm_grid_ids = pd.Index(in_scope_firms["grid_id_100m"].unique())
active_grid_ids = pd.Index(panel_active.unique()).union(firm_grid_ids)
unknown_firm_grids = firm_grid_ids.difference(raster["grid_id"])
if len(unknown_firm_grids):
    raise ValueError(f"{len(unknown_firm_grids)} assigned firm grid IDs are absent from the raster")

active = raster[raster["grid_id"].isin(active_grid_ids)].copy()
active["has_population"] = active["grid_id"].isin(pd.Index(panel_active.unique()))
active["has_firm"] = active["grid_id"].isin(firm_grid_ids)
centroids = active.geometry.centroid
active["centroid_x_3035"] = centroids.x
active["centroid_y_3035"] = centroids.y
routing_points = gpd.GeoSeries(centroids, crs=CRS_ANALYSIS).to_crs(CRS_ROUTING)
active["lon"] = routing_points.x
active["lat"] = routing_points.y

columns = ["grid_id", "municipality_id", "municipality_name", "centroid_x_3035", "centroid_y_3035", "lon", "lat", "has_population", "has_firm"]
output = active[columns].sort_values("grid_id").reset_index(drop=True)
if output["municipality_id"].isna().any() or len(output) != len(active_grid_ids):
    raise ValueError("Active-cell construction lost cells or municipality assignments")
output.to_parquet(OUTPUT_PATH, index=False)
pd.DataFrame({"metric": ["active cells", "population cells", "firm cells"], "count": [len(output), output.has_population.sum(), output.has_firm.sum()]})